# 10 - Quality Rule Validation

Validate the external rule-based quality grading layer. This notebook must not report supervised Grade A/B/C accuracy because the dataset has no expert grade labels.


In [5]:
from pathlib import Path
import importlib
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'src').exists() else NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.quality.evaluation as quality_evaluation
importlib.reload(quality_evaluation)
evaluate_weight_sets = quality_evaluation.evaluate_weight_sets

OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'quality_rule_eval'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)


Project root: /mnt/d/UWE/3rd Year/2nd semester/advanced_ai/ai_system


## Load custom image validation predictions

Use the custom validation table if available. Rows without model predictions are skipped.


In [6]:
validation_path = PROJECT_ROOT / 'docs' / 'report_figures' / 'custom_image_validation.csv'
predictions_path = PROJECT_ROOT / 'outputs' / 'xai_examples' / 'custom_image_tests' / 'custom_image_predictions.csv'

if validation_path.exists():
    validation_df = pd.read_csv(validation_path)
else:
    validation_df = pd.DataFrame()

if (validation_df.empty or 'quality_grade' not in validation_df.columns) and predictions_path.exists():
    print('Using custom_image_predictions.csv because validation CSV is missing quality fields.')
    validation_df = pd.read_csv(predictions_path)

display(validation_df.head())
print('Rows:', len(validation_df))


,image_id,image_path,expected_class,expected_produce_type,expected_freshness_status,model,predicted_class,predicted_produce_type,predicted_freshness_status,confidence,...,manual_review_required,reason_codes,quality_grade,overall_quality_score,component_scores,recommended_action,inventory_status,discount_percentage,warnings,notes
0,hea-app,/mnt/d/UWE/3rd Year/2nd semester/advanced_ai/a...,Apple__Healthy,Apple,healthy,EfficientNetB0 final,Apple__Healthy,Apple,healthy,0.989509,...,True,HEALTHY_PREDICTION|HIGH_CONFIDENCE|HIGH_CONFID...,A,90.2143,"{""model_condition"": 98.9509, ""color"": 98.4476,...",normal_sale,available,0.0,size_proxy_is_relative_to_image_area_not_physi...,NaN
1,hea-app,/mnt/d/UWE/3rd Year/2nd semester/advanced_ai/a...,Apple__Healthy,Apple,healthy,ResNet50 fine-tuned,Apple__Healthy,Apple,healthy,0.999710,...,True,HEALTHY_PREDICTION|HIGH_CONFIDENCE|HIGH_CONFID...,A,90.6733,"{""model_condition"": 99.971, ""color"": 98.4476, ...",normal_sale,available,0.0,size_proxy_is_relative_to_image_area_not_physi...,NaN
2,hea-app,/mnt/d/UWE/3rd Year/2nd semester/advanced_ai/a...,Apple__Healthy,Apple,healthy,MobileNetV2 fine-tuned,Apple__Healthy,Apple,healthy,0.981154,...,True,HEALTHY_PREDICTION|HIGH_CONFIDENCE|HIGH_CONFID...,A,89.8383,"{""model_condition"": 98.1154, ""color"": 98.4476,...",normal_sale,available,0.0,size_proxy_is_relative_to_image_area_not_physi...,NaN
3,hea-ban,/mnt/d/UWE/3rd Year/2nd semester/advanced_ai/a...,Banana__Healthy,Banana,healthy,EfficientNetB0 final,Banana__Healthy,Banana,healthy,0.999611,...,True,HEALTHY_PREDICTION|HIGH_CONFIDENCE|HIGH_CONFID...,A,91.1073,"{""model_condition"": 99.9611, ""color"": 99.0937,...",normal_sale,available,0.0,size_proxy_is_relative_to_image_area_not_physi...,NaN
4,hea-ban,/mnt/d/UWE/3rd Year/2nd semester/advanced_ai/a...,Banana__Healthy,Banana,healthy,ResNet50 fine-tuned,Banana__Healthy,Banana,healthy,1.000000,...,True,HEALTHY_PREDICTION|HIGH_CONFIDENCE|HIGH_CONFID...,A,91.1248,"{""model_condition"": 100.0, ""color"": 99.0937, ""...",normal_sale,available,0.0,size_proxy_is_relative_to_image_area_not_physi...,NaN


Rows: 33


## Rule-layer evaluation summaries

These summaries evaluate rule consistency and risk controls, not grade accuracy.


In [7]:
if validation_df.empty or 'quality_grade' not in validation_df.columns:
    print('Run custom_image_test.ipynb after quality integration to populate quality outputs.')
else:
    grade_distribution = (
        validation_df.groupby(['model', 'quality_grade'], dropna=False)
        .size()
        .reset_index(name='count')
    )
    manual_review_rate = (
        validation_df.groupby('model', dropna=False)['manual_review_required']
        .mean()
        .reset_index(name='manual_review_rate')
    )
    action_distribution = (
        validation_df.groupby(['model', 'recommended_action'], dropna=False)
        .size()
        .reset_index(name='count')
    )
    grade_distribution.to_csv(OUTPUT_DIR / 'grade_distribution.csv', index=False)
    manual_review_rate.to_csv(OUTPUT_DIR / 'manual_review_rate.csv', index=False)
    action_distribution.to_csv(OUTPUT_DIR / 'action_distribution.csv', index=False)
    display(grade_distribution)
    display(manual_review_rate)
    display(action_distribution)


,model,quality_grade,count
0,EfficientNetB0 final,A,4
1,EfficientNetB0 final,B,1
2,EfficientNetB0 final,C,5
3,EfficientNetB0 final,Review,1
4,MobileNetV2 fine-tuned,A,5
5,MobileNetV2 fine-tuned,B,1
6,MobileNetV2 fine-tuned,C,5
7,ResNet50 fine-tuned,A,4
8,ResNet50 fine-tuned,B,1
9,ResNet50 fine-tuned,C,6


,model,manual_review_rate
0,EfficientNetB0 final,0.727273
1,MobileNetV2 fine-tuned,0.727273
2,ResNet50 fine-tuned,0.727273


,model,recommended_action,count
0,EfficientNetB0 final,discount_or_quick_sale,1
1,EfficientNetB0 final,manual_review_before_listing,3
2,EfficientNetB0 final,manual_review_or_discard,2
3,EfficientNetB0 final,manual_review_required,1
4,EfficientNetB0 final,normal_sale,4
5,MobileNetV2 fine-tuned,discount_or_quick_sale,1
6,MobileNetV2 fine-tuned,manual_review_before_listing,3
7,MobileNetV2 fine-tuned,manual_review_or_discard,2
8,MobileNetV2 fine-tuned,normal_sale,5
9,ResNet50 fine-tuned,discount_or_quick_sale,1


## Weight sensitivity analysis

Compare `default_balanced`, `more_visual_colour`, and `safer_model_led`. This is not grade accuracy; it checks whether the rule layer is stable and avoids risky A/B decisions on rotten or uncertain predictions.


In [8]:
predictions_path = PROJECT_ROOT / 'outputs' / 'xai_examples' / 'custom_image_tests' / 'custom_image_predictions.csv'

if predictions_path.exists():
    predictions_df = pd.read_csv(predictions_path)
    weight_sensitivity = evaluate_weight_sets(predictions_df)
else:
    weight_sensitivity = pd.DataFrame(
        columns=[
            'weight_set',
            'grade_a_count',
            'grade_b_count',
            'grade_c_count',
            'review_count',
            'manual_review_rate',
            'rotten_to_c_or_review_rate',
            'risky_a_b_decisions',
            'note',
        ]
    )

weight_sensitivity.to_csv(OUTPUT_DIR / 'weight_sensitivity.csv', index=False)
display(weight_sensitivity)


,weight_set,grade_a_count,grade_b_count,grade_c_count,review_count,manual_review_rate,rotten_to_c_or_review_rate,risky_a_b_decisions,note
0,default_balanced,12,6,15,0,0.727273,1.0,0,No supervised grade labels; compare consistenc...
1,more_visual_colour,12,6,15,0,0.727273,1.0,0,No supervised grade labels; compare consistenc...
2,safer_model_led,11,7,15,0,0.727273,1.0,0,No supervised grade labels; compare consistenc...


## Formal quality-score ablation export

This section exports research evidence for the external rule-based grading layer. It compares risk behaviour across ablation profiles and does not report supervised Grade A/B/C accuracy.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.quality.evaluation import export_quality_score_ablation

written = export_quality_score_ablation()
written
